# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is. The functions below
are the ones behind the Monitoring page of the app.

To make sure there is something to show, this notebook first runs all six demo
scenarios in mock mode (approving only A and B, as the CLI demo does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

A completed


B completed
C blocked
D blocked
E blocked


F completed


In [3]:
from quantlab.monitoring import (
    capability_counts,
    guardrail_counts,
    kpis,
    read_events,
    recent_errors,
    stage_latency,
)

events = read_events(settings.events_path)
print(len(events), "events")
pd.Series(kpis(events))

105 events


n_runs               8.000
n_completed          5.000
n_blocked            3.000
n_failed             0.000
n_in_progress        0.000
success_rate         0.625
blocked_rate         0.375
failed_rate          0.000
n_replays            1.000
replay_match_rate    1.000
cache_hit_rate         NaN
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds):

In [4]:
pd.DataFrame(stage_latency(events))

,stage,count,mean_ms,p95_ms
0,planning,8,0.0,0.0
1,validation,8,0.2,0.3
2,execution,5,256.1,339.4
3,narration,5,0.0,0.0


Which capabilities run most, and which guardrails fire most:

In [5]:
pd.Series(capability_counts(events), name="executions")

load_series           7
resample              7
compute_returns       3
rolling_volatility    3
flag_regime           3
forward_return        3
compare_groups        3
ols_hac               3
spread                2
summary_stats         2
difference            2
adf_test              2
Name: executions, dtype: int64

In [6]:
pd.DataFrame(guardrail_counts(events))

,code,name,severity,count
0,G12,MATERIAL_ASSUMPTION,APPROVAL,5
1,R04,ALIGNMENT_LOSS,WARN,2
2,G10,LOOK_AHEAD,APPROVAL,1
3,G11,OVERLAPPING_OBSERVATIONS,WARN,1
4,G08,RETURNS_ON_RATES,BLOCK,1
5,G02,UNKNOWN_CAPABILITY,BLOCK,1
6,G13,PLANNER_REFUSED,BLOCK,1
7,G09,FREQUENCY_MISMATCH,BLOCK,1


The most recent failures (blocked runs and errors):

In [7]:
pd.DataFrame(recent_errors(events, limit=5))

,timestamp,run_id,event_type,error
0,2026-10-16T12:45:26.674Z,20261016-124526-bfbb5b,run_failed,The plan has blocking issues: G09:s7
1,2026-10-16T12:45:26.620Z,20261016-124526-33c104,run_failed,The plan has blocking issues: G13:plan
2,2026-10-16T12:45:26.567Z,20261016-124526-6deeb6,run_failed,"The plan has blocking issues: G08:s3, G02:s4"
